In [ ]:
from trustutils import run
run.TRUST_parameters()
run.introduction('MMS Trigonometric 3D', '2026')


In [ ]:
run.reset()

NB_PROCS = 2

case = run.addCase('.', 'MMS_Trig3D.data', nbProcs=NB_PROCS)
case.partition()          # trust -partition MMS_Trig3D.data 2 -> PAR_MMS_Trig3D.data + DOM_000*.Zones
# --- mesh-convergence scan (shared ladder: L1 = cells x2, L2 = cells x4) ---
# Levels instantiated from src/MMS_Trig3D.tpl.data (byte-copy of the baseline dataset
# with $nx/$ny/$nz node-count placeholders); the baseline case above is the
# untouched L0. Each level runs in its own build/scan/<lvl>/ directory so
# that the partitioned DOM_000*.Zones never overwrite another's.
NX0 = NY0 = NZ0 = 5                        # baseline node counts (L0)                       # baseline node counts (L0)
LEVELS = {
    "L1": {"nx": 2*NX0-1, "ny": 2*NY0-1, "nz": 2*NZ0-1},   # cells x2 -> h/2
    "L2": {"nx": 4*NX0-3, "ny": 4*NY0-3, "nz": 4*NZ0-3},   # cells x4 -> h/4
}
scan_cases = {}
for lvl, dic in LEVELS.items():
    c = run.addCaseFromTemplate("MMS_Trig3D.tpl.data", f"scan/{lvl}", dic=dic,
                                targetData=f"MMS_Trig3D_{lvl}.data",
                                nbProcs=NB_PROCS, excluNR=True)
    c.partition()          # trust -partition MMS_Trig3D_L1.data 2 -> PAR_*.data + Zones
    scan_cases[lvl] = c
run.printCases()
run.runCases()
run.tablePerf()

In [ ]:
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# The dataset as written is an invariance case: the uniform initial field
# b0 = (0, 1) satisfies u.grad(b) = 0 (u = 0 after the component-count fix),
# its Laplacian vanishes, and the walls hold (0, 1).  Both components must
# stay at their initial values to machine precision.
tbx, bx = read_son(case.dataFileName_, 'SONDE_BX')
tby, by = read_son(case.dataFileName_, 'SONDE_BY')

err_bx = np.max(np.abs(bx))
err_by = np.max(np.abs(by - 1.0))
print(f"max |Bx|          : {err_bx:.3e}")
print(f"max |By - 1|      : {err_by:.3e}")
print(f"final time        : {tbx[-1]:.6g}")
assert err_bx < 1e-10, f"Bx not preserved: {err_bx:.3e}"
assert err_by < 1e-10, f"By not preserved: {err_by:.3e}"
print("OK: uniform (0, 1) field is invariant under convection + diffusion.")


## Mesh convergence

The shared refinement ladder: `L0` = the baseline mesh above (`5×5×5`),
`L1` = cells ×2 (`9×9×9`), `L2` = cells ×4 (`17×17×17`).

The dataset is an **invariance case** (uniform (0, 1) field, convection on): the scan asserts the machine-precision identity at every level, no order reported.

In [ ]:


def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def level_error(subdir, case_name):
    """(max |Bx|, max |By - 1|) at the scan probes."""
    _, bx = read_son_dir(subdir, case_name, "SONDE_BX")
    _, by = read_son_dir(subdir, case_name, "SONDE_BY")
    return float(np.max(np.abs(bx))), float(np.max(np.abs(by - 1.0)))

scan = {"L0": ("", case.dataFileName_),
        "L1": (f"scan/L1", scan_cases["L1"].dataFileName_),
        "L2": (f"scan/L2", scan_cases["L2"].dataFileName_)}
scan_nodes = {"L0": (5, 5, 5), "L1": (9, 9, 9), "L2": (17, 17, 17)}
scan_err = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes      max |Bx|        max |By - 1|")
for lvl in ("L0", "L1", "L2"):
    nx, ny, nz = scan_nodes[lvl]
    ex, ey = scan_err[lvl]
    print(f"{lvl:<6} {nx:>3}x{ny}x{nz:<3} {ex:<14.3e} {ey:.3e}")
    assert ex < 1e-10, f"Bx not preserved at {lvl}: {ex:.3e}"
    assert ey < 1e-10, f"By not preserved at {lvl}: {ey:.3e}"
print("OK: uniform (0, 1) field invariance holds at every mesh level.")